# 08 — Creativity Metrics

Computes 6 creativity-focused text metrics per story, broken down by temperature schedule:

| # | Metric | Scope | Model |
|---|--------|-------|-------|
| 1 | **N-Gram Diversity** | per story | — |
| 2 | **Inverse Homogenization** | corpus (same prompt) | gte-large sentence emb |
| 3 | **Novelty** | corpus (all prompts) | spaCy GloVe word emb |
| 4 | **Surprise** | per story | spaCy GloVe word emb |
| 5 | **Lexical Complexity** | per story | — |
| 6 | **Syntactic Complexity** | per story | spaCy dep + benepar constituency |

**`semdis(a, b) = 1 − cosine_similarity(embed(a), embed(b))`**

Stories with the same `prompt_id` form one *item set* (used by InvHom).

In [3]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 120

from collections import Counter, defaultdict
from itertools import combinations
from tqdm.auto import tqdm

import nltk
nltk.download("punkt", quiet=True)

import config
from src.generation import load_stories

/Users/ishaicahila/Documents/תואר/NLP/nlp_final_project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

In [15]:
# ── Set to False to skip constituency parsing (benepar) for faster runs ──────
COMPUTE_CONSTITUENCY = False

MODEL = config.MODEL_NAME
print(f"Model: {MODEL}")
print(f"Constituency parsing: {COMPUTE_CONSTITUENCY}")

Model: Qwen/Qwen2.5-1.5B-Instruct
Constituency parsing: False


## Install / Verify Dependencies

```bash
pip install benepar
python -m spacy download en_core_web_lg
python -c "import benepar; benepar.download('benepar_en3')"
pip install sentence-transformers
```

In [16]:
import benepar; 
benepar.download('benpar_en3')

[nltk_data] Error loading benpar_en3: Package 'benpar_en3' not found
[nltk_data]     in index


False

In [17]:
import spacy
from sentence_transformers import SentenceTransformer

# ── spaCy en_core_web_lg: 300-d GloVe vectors for word-level semdis ──────────
nlp = spacy.load("en_core_web_lg")

if COMPUTE_CONSTITUENCY:
    import benepar
    if "benepar" not in nlp.pipe_names:
        nlp.add_pipe("benepar", config={"model": "benepar_en3"})

print(f"spaCy pipeline: {nlp.pipe_names}")

# ── Sentence embeddings: gte-large (for Inverse Homogenization) ───────────────
sentence_model = SentenceTransformer("thenlper/gte-large")
print("sentence-transformers: gte-large loaded")

spaCy pipeline: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 15083.68it/s]
BertModel LOAD REPORT from: thenlper/gte-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


sentence-transformers: gte-large loaded


## Load Stories

In [18]:
stories_by_schedule = {}
schedules_to_load = config.SCHEDULES + config.SWEEP_SCHEDULES
for sched_name in schedules_to_load:
    path = os.path.join(config.results_dir(MODEL, sched_name), "stories.jsonl")
    if os.path.exists(path):
        stories_by_schedule[sched_name] = load_stories(path)
        print(f"{sched_name}: {len(stories_by_schedule[sched_name])} stories")
    else:
        print(f"WARNING: {path} not found")

increasing: 50 stories
decreasing: 50 stories
valley: 50 stories
peak: 50 stories
fixed_0.5: 50 stories
fixed_0.7: 50 stories
fixed_1.0: 50 stories
fixed_1.2: 29 stories


In [19]:
# ── Flatten into corpus list ──────────────────────────────────────────────────
# item_set = prompt_id: all stories written on the same prompt form one item set.
corpus = []
for sched_name, stories in stories_by_schedule.items():
    for s in stories:
        corpus.append({
            "id":       f"{s['prompt_id']}_{sched_name}",
            "item_set": str(s["prompt_id"]),
            "schedule": sched_name,
            "text":     s["story"],
        })

print(f"Total stories in corpus : {len(corpus)}")
print(f"Unique item sets (prompts): {len(set(c['item_set'] for c in corpus))}")
print(f"Schedules: {list(stories_by_schedule.keys())}")

Total stories in corpus : 379
Unique item sets (prompts): 50
Schedules: ['increasing', 'decreasing', 'valley', 'peak', 'fixed_0.5', 'fixed_0.7', 'fixed_1.0', 'fixed_1.2']


## Helper Functions

In [20]:
# ── Dominant terms ─────────────────────────────────────────────────────────────
CONTENT_POS = {"NOUN", "VERB", "ADJ", "ADV"}

def get_dominant_terms(doc_or_span) -> list:
    """Lemmatized content words (N/V/Adj/Adv), excluding stopwords."""
    return [
        tok.lemma_.lower()
        for tok in doc_or_span
        if tok.pos_ in CONTENT_POS
        and not tok.is_stop
        and tok.is_alpha
        and not tok.is_space
    ]


# ── Word-level semdis via spaCy GloVe 300-d vectors ───────────────────────────
def word_vec(lemma: str) -> np.ndarray:
    """Return GloVe vector; zero vector for OOV."""
    return nlp.vocab[lemma].vector


def D_terms(terms: list) -> float:
    """
    D(S) = sum_{i!=j} semdis(T_i, T_j) / |T|

    sum_{i!=j} counts both orderings (i->j and j->i).
    Denominator is |T| (total terms, including OOV).
    OOV pairs (zero vectors) contribute 0 to the sum.
    """
    n = len(terms)
    if n < 2:
        return 0.0
    vecs = np.array([word_vec(t) for t in terms])          # [n, 300]
    norms = np.linalg.norm(vecs, axis=1)                    # [n]
    valid = norms > 1e-8
    n_valid = int(valid.sum())
    if n_valid < 2:
        return 0.0
    vn = vecs[valid] / norms[valid, None]                   # unit vectors [n_valid, 300]
    cos = vn @ vn.T                                         # [n_valid, n_valid]
    # sum_{i!=j} semdis = n_valid*(n_valid-1) - sum_{i!=j} cos_sim
    #   where sum_{i!=j} cos_sim = cos.sum() - trace = cos.sum() - n_valid
    semdis_sum = n_valid * (n_valid - 1) - (cos.sum() - n_valid)
    return float(semdis_sum / n)                            # divide by |T| (all terms)

In [21]:
# ── Syllable heuristic (English) ──────────────────────────────────────────────
def _syllables(word: str) -> int:
    word = word.lower().rstrip(".,!?;:\"'")
    if not word:
        return 0
    count, prev_v = 0, False
    for ch in word:
        v = ch in "aeiouy"
        if v and not prev_v:
            count += 1
        prev_v = v
    if word.endswith("e") and count > 1:
        count -= 1
    return max(1, count)


# ── Dependency chain depth ────────────────────────────────────────────────────
def _dep_depth(token) -> int:
    """Number of hops from token to root in the dependency tree."""
    d, cur = 0, token
    while cur.head != cur:
        cur = cur.head
        d += 1
    return d


# ── Constituency parse tree average leaf depth ────────────────────────────────
def _tree_avg_leaf_depth(parse_string: str) -> float:
    """Mean depth of leaf nodes in a benepar parse tree string."""
    try:
        tree = nltk.Tree.fromstring(parse_string)
        depths = [len(pos) for pos in tree.treepositions("leaves")]
        return float(np.mean(depths)) if depths else 0.0
    except Exception:
        return 0.0

## Per-Story Metric Functions

These operate on a single spaCy `Doc` object (already processed).

In [22]:
# ── Metric 1: N-Gram Diversity ────────────────────────────────────────────────
def ngram_diversity(text: str, ns=(1, 2, 3, 4, 5)) -> dict:
    """
    NGramDiv(S, n) = |unique n-grams| / |total n-grams|
    Uses spaCy tokenizer only (no tagger/parser needed).
    """
    doc = nlp.tokenizer(text)
    tokens = [t.text.lower() for t in doc if not t.is_space]
    result = {}
    for n in ns:
        grams = [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]
        result[n] = len(set(grams)) / len(grams) if grams else 0.0
    return result

In [23]:
# ── Metric 5: Lexical Complexity ──────────────────────────────────────────────
def lexical_complexity(doc) -> dict:
    """
    a. unique_words     — count of distinct word types
    b. avg_word_length  — mean characters per word token
    c. avg_sentence_length — mean unique words per sentence
    d. readability      — Flesch Reading Ease score
    """
    words = [t.text.lower() for t in doc if t.is_alpha and not t.is_space]
    sentences = list(doc.sents)

    unique_words     = len(set(words))
    avg_word_length  = float(np.mean([len(w) for w in words])) if words else 0.0

    if sentences:
        sent_unique = [
            len(set(t.text.lower() for t in s if t.is_alpha and not t.is_space))
            for s in sentences
        ]
        avg_sentence_length = float(np.mean(sent_unique))
    else:
        avg_sentence_length = 0.0

    n_sents     = max(len(sentences), 1)
    n_words     = max(len(words), 1)
    n_syllables = sum(_syllables(w) for w in words)
    readability = 206.835 - 1.015 * (n_words / n_sents) - 84.6 * (n_syllables / n_words)

    return {
        "unique_words":        unique_words,
        "avg_word_length":     avg_word_length,
        "avg_sentence_length": avg_sentence_length,
        "readability":         float(readability),
    }

In [24]:
# ── Metric 6: Syntactic Complexity ───────────────────────────────────────────
def syntactic_complexity(doc) -> dict:
    """
    a. pos_ratios            — avg per-sentence ratio for NOUN/ADJ/PRON/ADV
    b. avg_dep_path_length   — avg dependency chain hops to root
    c. avg_constituency_tree_depth — avg leaf depth in benepar parse tree
                                     (0.0 if COMPUTE_CONSTITUENCY=False)
    """
    TAG_SET = ["NOUN", "ADJ", "PRON", "ADV"]
    pos_ratios_per_sent = []
    dep_lengths         = []
    tree_depths         = []

    for sent in doc.sents:
        tokens = [t for t in sent if not t.is_space]
        if not tokens:
            continue

        n = len(tokens)
        counts = Counter(t.pos_ for t in tokens)
        pos_ratios_per_sent.append({tag: counts.get(tag, 0) / n for tag in TAG_SET})

        dep_lengths.extend(_dep_depth(t) for t in tokens)

        if COMPUTE_CONSTITUENCY:
            try:
                tree_depths.append(_tree_avg_leaf_depth(sent._.parse_string))
            except Exception:
                pass

    if pos_ratios_per_sent:
        avg_pos = {
            tag: float(np.mean([r[tag] for r in pos_ratios_per_sent]))
            for tag in TAG_SET
        }
    else:
        avg_pos = {tag: 0.0 for tag in TAG_SET}

    return {
        "pos_ratios":                   avg_pos,
        "avg_dep_path_length":           float(np.mean(dep_lengths)) if dep_lengths else 0.0,
        "avg_constituency_tree_depth":   float(np.mean(tree_depths)) if tree_depths else 0.0,
    }

In [25]:
# ── Metric 4: Surprise ────────────────────────────────────────────────────────
def surprise(doc) -> float:
    """
    Sur(S) = (2 / (|F|-1)) * sum_{i=2}^{|F|} |D(F_i) - D(F_{i-1})|

    F = sentences; D(F_i) = avg pairwise semdis of dominant terms in sentence F_i.
    Returns 0.0 if fewer than 2 sentences.
    """
    sentences = list(doc.sents)
    if len(sentences) < 2:
        return 0.0
    d_vals = [D_terms(get_dominant_terms(s)) for s in sentences]
    total  = sum(abs(d_vals[i] - d_vals[i-1]) for i in range(1, len(d_vals)))
    return float(2.0 / (len(sentences) - 1) * total)

## Corpus-Level Pre-computation

**Inverse Homogenization** requires sentence embeddings across all stories sharing the same `item_set`.  
**Novelty** requires *D(S_G)* — the corpus-wide average pairwise semdis of dominant terms — computed once.

In [26]:
# ── Sentence embeddings for Inverse Homogenization (gte-large) ───────────────
print("Encoding all stories with gte-large ...")

all_ids   = [s["id"]   for s in corpus]
all_texts = [s["text"] for s in corpus]

sent_embs = sentence_model.encode(
    all_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,   # pre-normalise → cosine via dot product
    convert_to_numpy=True,
)

id_to_emb = {sid: emb for sid, emb in zip(all_ids, sent_embs)}
print(f"Embeddings shape: {sent_embs.shape}")

Encoding all stories with gte-large ...


Batches: 100%|██████████| 12/12 [01:28<00:00,  7.34s/it]

Embeddings shape: (379, 1024)


In [15]:
# ── Metric 2: Inverse Homogenization ─────────────────────────────────────────
# Build lookup: item_set -> list of story ids in that set
_item_set_to_ids = defaultdict(list)
for s in corpus:
    _item_set_to_ids[s["item_set"]].append(s["id"])


def inv_homogenization(story_id: str, item_set: str) -> float:
    """
    InvHom(s|t) = (1/(|S_t|-1)) * sum_{s' in S_t \ {s}} semdis(s, s')

    semdis uses pre-normalised gte-large embeddings (semdis = 1 - dot product).
    Returns NaN if the story is the only one in its item set.
    """
    companion_ids = [i for i in _item_set_to_ids[item_set] if i != story_id]
    if not companion_ids:
        return float("nan")
    e_s   = id_to_emb[story_id]                                  # [d]
    e_set = np.array([id_to_emb[cid] for cid in companion_ids])  # [k, d]
    semdis_vals = 1.0 - (e_set @ e_s)                            # [k]
    return float(semdis_vals.mean())

<>:9: SyntaxWarning: invalid escape sequence '\ '
<>:9: SyntaxWarning: invalid escape sequence '\ '
/var/folders/6r/xnbdm8fs39j_xss3_ly98q680000gn/T/ipykernel_89191/2435487750.py:9: SyntaxWarning: invalid escape sequence '\ '
  """


In [27]:
# ── Pre-compute dominant terms for all stories (used by Novelty) ──────────────
# We use a lighter nlp pipeline (no benepar) since we only need tagger + lemmatizer.
nlp_light = spacy.load("en_core_web_lg", disable=["ner"])

print("Extracting dominant terms from corpus ...")
story_terms = {}   # story_id -> list[str]

for s in tqdm(corpus, desc="Dominant terms"):
    doc = nlp_light(s["text"])
    story_terms[s["id"]] = get_dominant_terms(doc)

# Corpus-level term list T_G (all terms concatenated, with repetition)
all_corpus_terms = [t for terms in story_terms.values() for t in terms]
print(f"Total corpus terms |T_G|: {len(all_corpus_terms)}")

# D(S_G) — computed once and reused for every story's Novelty score
print("Computing D(S_G) — may take a few minutes for large corpora ...")
D_G = D_terms(all_corpus_terms)
print(f"D(S_G) = {D_G:.6f}")

Extracting dominant terms from corpus ...


Dominant terms: 100%|██████████| 379/379 [00:17<00:00, 21.66it/s]


Total corpus terms |T_G|: 100719
Computing D(S_G) — may take a few minutes for large corpora ...


: 

In [1]:
# ── Metric 3: Novelty ─────────────────────────────────────────────────────────
def novelty(story_id: str) -> float:
    """
    Nov(S_n) = 2 * |D(S_n) - D(S_G)|    range: [0, 2]
    """
    d_sn = D_terms(story_terms[story_id])
    return float(2.0 * abs(d_sn - D_G))

## Compute All Metrics

> **Performance note:** The full spaCy pipeline with benepar runs ~1–3 s/story.  
> For 200 stories expect ~5–10 min. Set `COMPUTE_CONSTITUENCY = False` to skip benepar.

In [48]:
raw_results = []   # list of per-story result dicts

for s in tqdm(corpus, desc="Computing metrics"):
    story_id = s["id"]
    item_set = s["item_set"]
    text     = s["text"]

    # Full spaCy parse (includes benepar if COMPUTE_CONSTITUENCY=True)
    doc = nlp(text)

    raw_results.append({
        "id":                  story_id,
        "schedule":            s["schedule"],
        "item_set":            item_set,
        "ngram_diversity":     ngram_diversity(text),
        "inv_homogenization":  inv_homogenization(story_id, item_set),
        "novelty":             novelty(story_id),
        "surprise":            surprise(doc),
        "lexical_complexity":  lexical_complexity(doc),
        "syntactic_complexity": syntactic_complexity(doc),
    })

print(f"Done — {len(raw_results)} stories processed.")

Computing metrics: 100%|██████████| 200/200 [00:21<00:00,  9.21it/s]

Done — 200 stories processed.


## Results

In [49]:
# ── Flatten nested dicts into wide DataFrame ──────────────────────────────────
rows = []
for r in raw_results:
    row = {
        "id":                  r["id"],
        "schedule":            r["schedule"],
        "item_set":            r["item_set"],
        "inv_homogenization":  r["inv_homogenization"],
        "novelty":             r["novelty"],
        "surprise":            r["surprise"],
    }
    for n, v in r["ngram_diversity"].items():
        row[f"ngdiv_{n}"] = v
    row.update({f"lex_{k}": v for k, v in r["lexical_complexity"].items()})
    row["syn_avg_dep_path"]   = r["syntactic_complexity"]["avg_dep_path_length"]
    row["syn_avg_tree_depth"] = r["syntactic_complexity"]["avg_constituency_tree_depth"]
    for pos, val in r["syntactic_complexity"]["pos_ratios"].items():
        row[f"syn_pos_{pos.lower()}"] = val
    rows.append(row)

df = pd.DataFrame(rows)
print(f"DataFrame shape: {df.shape}")
df

DataFrame shape: (200, 21)


,id,schedule,item_set,inv_homogenization,novelty,surprise,ngdiv_1,ngdiv_2,ngdiv_3,ngdiv_4,...,lex_unique_words,lex_avg_word_length,lex_avg_sentence_length,lex_readability,syn_avg_dep_path,syn_avg_tree_depth,syn_pos_noun,syn_pos_adj,syn_pos_pron,syn_pos_adv
0,0_increasing,increasing,0,0.114746,80585.272888,159.836791,0.959036,1.000000,1.000000,1.0,...,387,6.185930,130.666667,-99.572420,11.546988,0.0,0.155502,0.098068,0.117341,0.153120
1,1_increasing,increasing,1,0.135254,80491.958649,144.080557,0.967105,0.997802,1.000000,1.0,...,434,6.209459,145.666667,-108.583649,8.684211,0.0,0.242945,0.108466,0.057319,0.086714
2,2_increasing,increasing,2,0.134155,80674.426666,51.420952,0.915531,1.000000,1.000000,1.0,...,322,5.609329,56.166667,-4.850391,4.397820,0.0,0.241521,0.108243,0.053891,0.195303
3,3_increasing,increasing,3,0.117371,80502.290222,104.217709,0.935065,1.000000,1.000000,1.0,...,422,6.240363,107.750000,-69.856505,5.303030,0.0,0.228732,0.069295,0.093738,0.124045
4,4_increasing,increasing,4,0.122864,80469.592499,200.158970,0.987835,1.000000,1.000000,1.0,...,402,5.913580,134.666667,-76.203333,6.807786,0.0,0.234487,0.101790,0.063313,0.091464
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,45_peak,peak,45,0.129883,80525.886108,36.578623,0.899782,0.982533,0.997812,1.0,...,404,5.661939,46.333333,11.330000,4.200436,0.0,0.212526,0.085021,0.061962,0.146405
196,46_peak,peak,46,0.136719,80512.418610,56.438854,0.967517,1.000000,1.000000,1.0,...,408,5.642686,51.875000,12.928125,4.389791,0.0,0.269136,0.089529,0.053097,0.078021
197,47_peak,peak,47,0.104675,80500.251892,124.461809,0.975845,1.000000,1.000000,1.0,...,394,6.558313,132.333333,-102.911844,8.500000,0.0,0.284268,0.172502,0.025341,0.089847
198,48_peak,peak,48,0.118835,80549.919098,170.072811,0.970917,1.000000,1.000000,1.0,...,427,5.938356,144.333333,-102.828973,6.167785,0.0,0.232128,0.095155,0.090305,0.113483


In [50]:
# ── Summary by schedule ───────────────────────────────────────────────────────
metric_cols = [c for c in df.columns if c not in {"id", "schedule", "item_set"}]
summary = df.groupby("schedule")[metric_cols].agg(["mean", "std"]).round(4)
summary

inv_homogenization             novelty          surprise            \
                         mean     std        mean      std     mean       std   
schedule                                                                        
decreasing             0.1303  0.0224  80559.7767  94.5473  84.1030  108.2038   
increasing             0.1260  0.0222  80544.3587  75.9257  68.6247   51.3626   
peak                   0.1253  0.0244  80548.2624  93.5080  78.1705   76.0878   
valley                 0.1291  0.0249  80556.8331  91.7229  78.6221   69.3970   

           ngdiv_1         ngdiv_2          ... syn_avg_tree_depth       \
              mean     std    mean     std  ...               mean  std   
schedule                                    ...                           
decreasing  0.9222  0.0507  0.9951  0.0083  ...                0.0  0.0   
increasing  0.9323  0.0356  0.9962  0.0057  ...                0.0  0.0   
peak        0.9278  0.0468  0.9964  0.0088  ...                0.0  0.0   
valley      0.9372  0.0392  0.9966  0.0103  ...                0.0  0.0   

           syn_pos_noun         syn_pos_adj         syn_pos_pron          \
                   mean     std        mean     std         mean     std   
schedule                                                                   
decreasing       0.2104  0.0725      0.0949  0.0397       0.0829  0.0311   
increasing       0.2091  0.0543      0.0877  0.0213       0.0786  0.0238   
peak             0.2287  0.0634      0.0909  0.0316       0.0745  0.0217   
valley           0.2167  0.0389      0.0983  0.0298       0.0751  0.0294   

           syn_pos_adv          
                  mean     std  
schedule                        
decreasing      0.1132  0.0351  
increasing      0.1059  0.0311  
peak            0.1074  0.0268  
valley          0.1181  0.0338  

[4 rows x 36 columns]

In [ ]:
# ── Min-max normalize all metric columns to [0, 2] ───────────────────────────
# Enables direct cross-metric comparison on a common scale.
def minmax_scale_02(series: pd.Series) -> pd.Series:
    lo, hi = series.min(), series.max()
    if hi - lo < 1e-12:                       # constant column → midpoint
        return pd.Series(np.ones(len(series)), index=series.index)
    return 2.0 * (series - lo) / (hi - lo)

df_norm = df.copy()
for col in metric_cols:
    if df_norm[col].notna().any():
        df_norm[col] = minmax_scale_02(df_norm[col].fillna(df_norm[col].median()))

print("All metric columns normalized to [0, 2]")
df_norm.groupby("schedule")[metric_cols].mean().round(3)


## Visualisations

In [ ]:
# ── Bar plots: main creativity metrics by schedule (normalized to [0, 2]) ─────
MAIN_METRICS = {
    "ngdiv_1":            "Unigram Diversity",
    "ngdiv_2":            "Bigram Diversity",
    "inv_homogenization": "Inv. Homogenization (↑ = more unique)",
    "novelty":            "Novelty",
    "surprise":           "Surprise",
    "lex_readability":    "Flesch Readability",
    "syn_avg_dep_path":   "Avg Dep Path Length",
    "syn_avg_tree_depth": "Avg Parse Tree Depth",
}

scheds = sorted(df_norm["schedule"].unique())
x = np.arange(len(scheds))
COLORS = plt.cm.tab10.colors

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, (col, title) in zip(axes.flat, MAIN_METRICS.items()):
    means = [df_norm.loc[df_norm["schedule"] == s, col].mean() for s in scheds]
    stds  = [df_norm.loc[df_norm["schedule"] == s, col].std()  for s in scheds]
    ax.bar(x, means, yerr=stds, capsize=4,
           color=COLORS[:len(scheds)], alpha=0.85, ecolor="grey")
    ax.set_xticks(x)
    ax.set_xticklabels(scheds, rotation=30, ha="right", fontsize=8)
    ax.set_ylim(0, 2)
    ax.set_title(title, fontsize=9)

plt.suptitle("Creativity Metrics by Temperature Schedule (normalized [0, 2])", fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(config.RESULTS_DIR, "creativity_metrics_bar.png"),
            dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── N-Gram Diversity across n-orders, one line per schedule (normalized [0, 2]) ─
ns = [1, 2, 3, 4, 5]
fig, ax = plt.subplots(figsize=(8, 4))

for i, sched in enumerate(scheds):
    sub   = df_norm[df_norm["schedule"] == sched]
    means = [sub[f"ngdiv_{n}"].mean() for n in ns]
    stds  = [sub[f"ngdiv_{n}"].std()  for n in ns]
    ax.plot(ns, means, marker="o", label=sched, color=COLORS[i])
    ax.fill_between(ns,
                    np.array(means) - np.array(stds),
                    np.array(means) + np.array(stds),
                    alpha=0.12, color=COLORS[i])

ax.set_xlabel("n (gram order)")
ax.set_ylabel("Diversity (normalized [0, 2])")
ax.set_ylim(0, 2)
ax.set_title("N-Gram Diversity by Order and Schedule")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(config.RESULTS_DIR, "ngram_diversity_by_order.png"), dpi=150)
plt.show()


In [ ]:
# ── POS Ratio bars (4 tags × schedules, normalized [0, 2]) ───────────────────
pos_tags = ["NOUN", "ADJ", "PRON", "ADV"]
fig, axes = plt.subplots(1, 4, figsize=(14, 4))

for ax, pos in zip(axes, pos_tags):
    col   = f"syn_pos_{pos.lower()}"
    means = [df_norm.loc[df_norm["schedule"] == s, col].mean() for s in scheds]
    stds  = [df_norm.loc[df_norm["schedule"] == s, col].std()  for s in scheds]
    ax.bar(x, means, yerr=stds, capsize=3,
           color=COLORS[:len(scheds)], alpha=0.85, ecolor="grey")
    ax.set_xticks(x)
    ax.set_xticklabels(scheds, rotation=30, ha="right", fontsize=8)
    ax.set_ylim(0, 2)
    ax.set_title(f"POS ratio: {pos}", fontsize=9)

plt.suptitle("Avg POS Ratios by Schedule (normalized [0, 2])", fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(config.RESULTS_DIR, "pos_ratios_by_schedule.png"),
            dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Box plots: Novelty and Surprise distributions (normalized [0, 2]) ────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

for ax, col, title in [
    (ax1, "novelty",  "Novelty"),
    (ax2, "surprise", "Surprise"),
]:
    data = [df_norm.loc[df_norm["schedule"] == s, col].dropna().values for s in scheds]
    bp = ax.boxplot(data, patch_artist=True, showfliers=False,
                    medianprops=dict(color="black", linewidth=1.5))
    for patch, color in zip(bp["boxes"], COLORS):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)
    ax.set_xticks(range(1, len(scheds)+1))
    ax.set_xticklabels(scheds, rotation=30, ha="right", fontsize=8)
    ax.set_ylim(0, 2)
    ax.set_title(title)

plt.suptitle("Distribution of Novelty and Surprise by Schedule (normalized [0, 2])", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(config.RESULTS_DIR, "novelty_surprise_box.png"), dpi=150)
plt.show()


In [55]:
# ── Save full results to CSV ──────────────────────────────────────────────────
out_path = os.path.join(config.RESULTS_DIR, "creativity_metrics.csv")
df.to_csv(out_path, index=False)
print(f"Saved → {out_path}")
print(df.describe().round(4))

Saved → /Users/ishaicahila/Documents/תואר/NLP/nlp_final_project/notebooks/../results_with_500/creativity_metrics.csv
       inv_homogenization     novelty  surprise   ngdiv_1   ngdiv_2   ngdiv_3  \
count            200.0000    200.0000  200.0000  200.0000  200.0000  200.0000   
mean               0.1277  80552.3077   77.3801    0.9299    0.9961    0.9996   
std                0.0234     88.7934   78.5811    0.0435    0.0084    0.0040   
min                0.0771  80358.8419    5.7006    0.7411    0.9320    0.9448   
25%                0.1115  80494.8645   32.4739    0.9134    0.9957    1.0000   
50%                0.1254  80525.9006   55.3894    0.9412    1.0000    1.0000   
75%                0.1412  80583.4791   94.2368    0.9598    1.0000    1.0000   
max                0.1940  80832.3141  555.7555    0.9878    1.0000    1.0000   

        ngdiv_4   ngdiv_5  lex_unique_words  lex_avg_word_length  \
count  200.0000  200.0000          200.0000             200.0000   
mean     0.9998  